# Lasso Regression Research Project

Note: Under progress

This projetc is an exposition of the theory of the Lasso linear regression model, including implementation using coordinate descent optimisation, and an empirical validation of certain theoretical high probability inequalities

Based on cambridge course: **Modern Statistical Methods** by R. Shah, part iii mathematics Uni of Cambridge. See references at end

## 0. Introducing the Lasso

Let us recall the general statistical learning problem at hand:

Suppose we have $(\mathbf{x}_i\times{y_i})_{i=1}^n\overset{iid}{\sim}\overset{n}{\underset{i=1}{\bigotimes}}\,P(\mathbb{R}^p\times{\mathbb{R}})$ training data. Consider using a MSE error loss function $l:\mathbb{R}^p\times{\mathbb{R}}\times{\mathcal{H}}\rightarrow{\mathbb{R}^{\geq{0}}}, (X,Y,f)\mapsto{||Y-f(X)||_2^2}$ where $\mathcal{H}$ is some suitably large hypothesis class of functions. We wish to find:
$$f_{\mathcal{H}}^*=\underset{f\in{\mathcal{H}}}{arg\,min}\:R(f)\in{\mathcal{H}}$$
where $R(f)=\mathbb{E}_{X\times{Y}\sim{P}}\big(l(X,Y,f)\big)$ is the population risk. To minimise $R$ over $f\in{\mathcal{H}}$, by conditioning, is to minimise the integral:
$$\int_{\mathbb{R}^p}\int_{\mathbb{R}}||Y-f(X)||_2^2d\mathbb{P}_{Y|X}d\mathbb{P}_X=\mathbb{E}_{X\sim{\mathbb{P}_X}}\mathbb{E}_{Y|X\sim{\mathbb{P}_{Y|X}}}\big(||Y-f(X)||_2^2\big|X=x)$$

Hence, writing $f(x)=f_x$ for each $x\in{\mathbb{R}^p}$, it suffices to $R(f)$ by minimising the following conditional expectation, pointwise in $x$
$$\mathbb{E}_{Y|X\sim{\mathbb{P}_{Y|X}}}\big(||Y-f_x||_2^2\big|X=x)$$
Since this is a convex and differentiable function in $f_x\in{\mathbb{R}}$, we can differentiate with respect to $f_x$ to find the global minima as the conditional expectation:
$$f(x)=\mathbb{E}_{Y|X\sim{\mathbb{P}_{Y|X}}}\big(Y|X=x\big)$$
So, the regression function $f$ is minimised by outputting the mean of $Y$ as a random variable conditioned against $X=x$.

Now, when performing linear regression for our training data, we make the hypothesis that our MSE population risk minimiser, namely the coniditional mean, takes a linear form
$$f_{\mathcal{H}}^*(X)=\mathbb{E}_{Y|X\sim{\mathbb{P}_{Y|X}}}\big(Y|X=x\big)=X\beta + \epsilon$$
where $\epsilon\sim{\mathcal{N}(0,\sigma)}$, $\beta\in{\mathbb{R}^p}$ and $X=(\mathbf{x}_1,...\mathbf{x}_n)^T\in{\mathbb{R}^{n\times{p}}}$. Where we re-initialise $\mathbf{x}_i=(1,x_{i,j})_{j=1}^p$ and $p=p+1$, so that $\beta_0$ is the intercept.

## 1. Empirical risk, sparsity and the Lasso

We want to minimise the **empirical risk** of our linear model $X\beta$ under mean squared error (MSE), with an added $\ell_1$ penalty. Let us build that objective from the training pairs in Section 0.

### 1.1 From training pairs to empirical MSE

For each training observation $i$, we have a pair $(\mathbf x_i,y_i)$, where $\mathbf x_i=(x_{i1},\dots,x_{ip})^\top\in\mathbb R^p$ and $y_i\in\mathbb R$. A **predictor** is an input variable used to make a prediction. For example, when predicting house prices, floor area and number of bedrooms can be predictors. The **response** is the quantity we want to predict, such as the observed sale price $y_i$.

For now write $X^{\rm raw}$ for the $n\times p$ matrix of the original predictors. Row $i$ is $\mathbf x_i^\top$, column $j$ contains all $n$ observations of predictor $j$, and $y^{\rm raw}=(y_1,\ldots,y_n)^\top$ collects the responses. As in Section 0, we can put a column of ones in the design matrix to represent the intercept. Here we write the intercept separately as $a\in\mathbb R$, so our fitted value at observation $i$ is
$$
\hat y_i(a,b)=a+\mathbf x_i^\top b
=a+\sum_{j=1}^p x_{ij}b_j,
\qquad b\in\mathbb R^p.
$$
The **residual** at this observation is $r_i(a,b)=y_i-\hat y_i(a,b)$. For example, if the observed response is $10$ and the prediction is $8$, the residual is $2$. A residual is computed from a chosen model and the observed data. In the statistical model $Y=a^0+X^\top b^0+\epsilon$ for one observation, the noise $\epsilon$ is the deviation from the true conditional mean, which we generally cannot observe directly.

Section 0 defined population risk by averaging over the unknown distribution of $(X,Y)$. Empirical risk averages the same squared loss over the $n$ training observations:
$$
\widehat R_n(a,b)
=\frac1n\sum_{i=1}^n \big(y_i-a-\mathbf x_i^\top b\big)^2
=\frac1n\|y^{\rm raw}-a\mathbf 1-X^{\rm raw}b\|_2^2.
$$
Ordinary least squares (OLS) chooses $(a,b)$ to minimise $\widehat R_n(a,b)$.

### 1.2 Why add the Lasso penalty?

In Shah Chapter 2, Section 2.1, the starting assumption is that there may be $p$ measured predictors, while only a small number enter the true linear mean. If $b^0$ is the true slope vector, define
$$
S=\{j:b^0_j\ne0\},\qquad s=|S|.
$$
We call $b^0$ **sparse** when $s\ll p$. In a fixed, full-column-rank design with noise variance $\sigma^2$, Shah compares the OLS mean squared prediction error for the true mean at the training design, $\sigma^2p/n$, with $\sigma^2s/n$ for a fit that already knows which $s$ columns to use. The comparison explains the potential benefit of using fewer predictors. In practice, $S$ is unknown.

Shah Section 2.2, equation (2.1), introduces Lasso by adding the $\ell_1$ norm of the slopes, $\|b\|_1=\sum_{j=1}^p |b_j|$. Using the predictor units in which we intend to fit, the objective is
$$
\min_{a,b}\left\{
\frac{1}{2n}\sum_{i=1}^n\big(y_i-a-\mathbf x_i^\top b\big)^2
+\lambda\sum_{j=1}^p|b_j|
\right\}.
$$
The first term is half the empirical MSE. The factor $1/2$ cancels the $2$ when we differentiate. The second term charges $\lambda|b_j|$ for using slope $j$. The intercept $a$ is free to set the overall level of the predictions. At $\lambda=0$ the criterion reduces to least squares. For $\lambda>0$, Lasso can shrink a fitted slope all the way to zero. A zero slope means its predictor contributes nothing to the fitted linear mean. The KKT conditions below show precisely when zero is allowed.

### 1.3 Centre first, then scale

Before fitting, we centre and scale each **predictor column** using its own training mean and training standard deviation. For predictor $j$, compute
$$
m_j=\frac1n\sum_{i=1}^n x_{ij},
\qquad
d_j=\sqrt{\frac1n\sum_{i=1}^n(x_{ij}-m_j)^2},
\qquad
z_{ij}=\frac{x_{ij}-m_j}{d_j}.
$$
**Centring** is subtracting $m_j$. It moves the column's mean to zero. **Scaling** is dividing the centred values by $d_j$. It expresses them in units of one training standard deviation. For example, if training floor areas have mean $80$ and standard deviation $20$ square metres, an area of $110$ is centred to $30$ and then scaled to $1.5$. The same training values $80$ and $20$ are used for validation and test rows. A constant predictor becomes a column of zeros after centring, and the code uses a divisor of $1$ for that column.

Scaling matters because the Lasso penalty acts on coefficient magnitudes. Measuring floor area in square centimetres would make the raw coefficient much smaller than measuring it in square metres, even for the same predictions. We therefore penalise slopes in standardised predictor units.

Let $z_i^\top$ denote row $i$ of the standardised design and let $\beta$ be the vector of slopes in these units. For any fixed $\beta$, the intercept in the model $a+z_i^\top\beta$ minimises the squared error when
$$
0=-\frac1n\sum_{i=1}^n(y_i-a-z_i^\top\beta)
\quad\Longrightarrow\quad
a=\bar y-\bar z^\top\beta=\bar y,
$$
because each standardised column has mean zero. Thus we subtract the training mean from the response and fit the slopes to $y_i-\bar y$. We leave the response in its original units. To predict a new response, standardise its predictors using the training statistics and add $\bar y$ back to the model output.

For the rest of the notebook, $X$ denotes the centred, scaled design and $y$ denotes the centred response. The slope-only problem is
$$
\hat\beta_\lambda\in\operatorname*{arg\,min}_{\beta\in\mathbb R^p}
Q_\lambda(\beta),
\qquad
Q_\lambda(\beta)=\underbrace{\frac1{2n}\|y-X\beta\|_2^2}_{\text{half the empirical MSE}}
+\underbrace{\lambda\|\beta\|_1}_{\text{Lasso penalty}}.
$$
This is the centred form of Shah Section 2.2, equation (2.2). ESL Chapter 3, Section 3.4.2, also gives a constrained formulation using an $\ell_1$ budget.

### 1.4 Convex optimality and the KKT conditions

The squared-error term and the $\ell_1$ penalty are both convex, so their sum is convex. The loss is differentiable in $\beta$, with gradient
$$
-\frac{X^\top(y-X\beta)}n.
$$
The penalty involves $|t|$, whose slope is $-1$ for $t<0$ and $+1$ for $t>0$. At $t=0$ there is a corner. A **subgradient** at a point is a slope of a line through that point that stays below the function. At zero, $|u|\ge vu$ for every $u\in\mathbb R$ exactly when $v\in[-1,1]$. Thus
$$
\partial |t|=
\begin{cases}
\{-1\}, & t<0,\\
[-1,1], & t=0,\\
\{1\}, & t>0.
\end{cases}
$$
The **convex optimality condition** is
$$
\hat\beta\text{ minimises }Q_\lambda
\quad\Longleftrightarrow\quad
0\in\partial Q_\lambda(\hat\beta).
$$
The zero subgradient gives a horizontal supporting plane, so every other point has objective value at least $Q_\lambda(\hat\beta)$. See Shah Section 2.2.3, Propositions 19 and 20.

To apply this condition, let $r=y-X\hat\beta$ be the fitted residual vector and $X_{\cdot j}$ be column $j$ of $X$. Set $g_j=X_{\cdot j}^\top r/n$. Then $-g_j$ is the derivative of the squared-error term with respect to slope $j$. We need a subgradient $v_j$ of the absolute-value term such that $-g_j+\lambda v_j=0$. Substituting the three cases above gives the **Lasso KKT conditions** in Shah Section 2.2.4:
$$
\begin{aligned}
\hat\beta_j>0 &\ \Longrightarrow\ g_j=\lambda,\\
\hat\beta_j<0 &\ \Longrightarrow\ g_j=-\lambda,\\
\hat\beta_j=0 &\ \Longrightarrow\ |g_j|\le\lambda.
\end{aligned}
$$
At a zero slope, the penalty can balance any loss derivative between $-\lambda$ and $\lambda$. For example, if $\lambda=0.3$ and $g_j=0.1$, the choice $v_j=g_j/\lambda=1/3$ is allowed at zero and cancels the derivative. This is how the $\ell_1$ penalty can produce exact zeros. At a nonzero slope the penalty has a single possible derivative, so the score must equal the corresponding endpoint.

A candidate vector solves the Lasso problem if it satisfies these conditions for every coordinate. For the all-zero vector, $r=y$, so the smallest penalty at which all slopes can be zero is $\lambda_{\max}=\|X^\top y/n\|_\infty$. The next code cell evaluates the objective, computes the largest KKT violation, and defines the centring and scaling used in the experiments.

In [ ]:
def objective(X, y, beta, lam):
    return 0.5 * np.mean((y - X @ beta)**2) + lam * np.sum(np.abs(beta))


def kkt_error(X, y, beta, lam):
    score = X.T @ (y - X @ beta) / len(y)
    error = np.maximum(np.abs(score) - lam, 0)
    active = beta != 0
    error[active] = np.abs(score[active] - lam * np.sign(beta[active]))
    return error.max()


def standardise(X):
    mean = X.mean(axis=0)
    scale = X.std(axis=0)
    scale[scale == 0] = 1
    return (X - mean) / scale, mean, scale


def lambda_max(X, y):
    return np.max(np.abs(X.T @ y)) / len(y)

## References

1. Rajen D. Shah, *Modern Statistical Methods*, supplied file **Modern Statistical Methods R.Shah.pdf**, Chapter 2. [Modern Statistical Methods](https://www.dpmms.cam.ac.uk/~rds37/teaching/modern_stat_methods/notes.pdf).
2. Trevor Hastie, Robert Tibshirani and Jerome Friedman (2009), *The Elements of Statistical Learning*, 2nd ed., [Publisher chapter](https://doi.org/10.1007/978-0-387-84858-7_3).